<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/LangGraph_MultiAgent_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph Multi-Agent Practical — Customer Support Triage

**Companion notebook for the session:** *LangGraph Multi-Agent Design — Node Design, Agent Roles, Communication Protocols*

You will build a small team of agents that handles a customer support ticket:

```
                  ┌──────────────┐
              ┌──►│ billing_agent├──┐   (dashed = Command handoff)
              │   └──────────────┘  ┆
START ──► triage                    ┆──► escalation_agent ──► responder ──► END
              │   ┌──────────────┐  ┆                             ▲
              └──►│  tech_agent  ├──┘                             │
                  └──────────────┘──────────────────────────────►─┘
```

**Everything runs on the free Hugging Face Inference API.** No credit card, no paid key.

### What each part of the session maps to

| Session slide | Where you see it here |
|---|---|
| Slide 4 — State & reducers | Step 2 |
| Slide 5 — The 4 beats of a node | Step 4 |
| Slide 7 — Conditional edges | Step 6 |
| Slide 9 — Role prompts | Step 3 |
| Slide 11 — `Command` handoff | Step 7 |
| Slide 14 — Loops, threads, control | Step 11 |

### Before you start

1. Create a **free** account at <https://huggingface.co>.
2. Go to **Settings → Access Tokens → Create new token**, choose **Fine-grained**, and tick
   **"Make calls to Inference Providers"**.
3. In Google Colab: click the 🔑 key icon in the left sidebar → **Add new secret** →
   name it `HF_TOKEN`, paste your token, and enable notebook access.
   Running locally? Just set the environment variable `HF_TOKEN`.

> **No token yet, or hit a rate limit mid-class?** Set `MOCK_MODE = True` in Step 1.
> Every agent then returns a canned answer. The graph, the routing, the handoffs and the
> trace all still work — you just do not get real model text. Nothing else changes.

---
## Step 1 — Install and connect to a free LLM

Two libraries: `langgraph` (the graph engine) and `huggingface_hub` (the free LLM client).

In [ ]:
%pip install -q -U langgraph huggingface-hub<2.0,>=0.25.0
print("installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
installed


In [ ]:
import os, getpass

# ── Flip this to True to run the whole notebook with NO token and NO internet ──
MOCK_MODE = False

HF_TOKEN = None
if not MOCK_MODE:
    try:                                     # Google Colab secrets
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
    HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Paste your Hugging Face token (or press Enter to use MOCK_MODE): ").strip()
    if not HF_TOKEN:
        MOCK_MODE = True

print("MOCK_MODE =", MOCK_MODE)

In [ ]:
# A free, small, instruction-following model. If one is unavailable, the helper below
# automatically tries the next candidate in this list.
MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "HuggingFaceTB/SmolLM3-3B",
]

# Tip: to see everything currently served for free, run in a cell:
#   !hf models ls --warm | head -40

### The `chat()` helper — the ONLY place the notebook talks to an LLM

Every agent calls this one function. That is deliberate: swapping model or provider later
means editing exactly one cell.

`InferenceClient.chat.completions.create(...)` is the same shape as the OpenAI SDK, so this
code is portable.

In [ ]:
MOCK_REPLIES = {
    "billing":    "Our policy allows a full refund for a duplicate charge within 30 days. "
                  "I can see two identical charges on your account.\n"
                  "CONFIDENCE: LOW",
    "technical":  "Clear the app cache and reinstall the latest version. If the crash persists, "
                  "export your logs from Settings > About > Export logs and send them over.\n"
                  "CONFIDENCE: HIGH",
    "escalation": "Summary: customer issue needs a human decision.\n"
                  "Recommended action: route to the Tier-2 queue for this topic.\n"
                  "Target response time: 4 hours.",
    "the final reply":
                  "Hi, thanks for getting in touch. We have looked into this and our team is "
                  "on it - you will hear back from us shortly with a resolution.",
}


def _mock_classify(user: str) -> str:
    """Pretend to be an LLM classifier, using keywords instead of a model."""
    low = user.lower()
    if any(w in low for w in ("charge", "refund", "invoice", "bill", "payment", "plan")):
        category = "billing"
    elif any(w in low for w in ("crash", "bug", "login", "error", "slow", "app", "screen")):
        category = "technical"
    else:
        category = "other"
    urgent = any(w in low for w in ("refund", "manager", "urgent", "twice", "blocked", "human"))
    return '{"category": "%s", "urgency": "%s"}' % (category, "high" if urgent else "medium")


def _mock(system: str, user: str) -> str:
    low = system.lower()
    if "classifier" in low:
        return _mock_classify(user)
    for key, reply in MOCK_REPLIES.items():
        if key in low:
            return reply
    return "Thanks for contacting support. We are looking into it."


if not MOCK_MODE:
    from huggingface_hub import InferenceClient
    _client = InferenceClient(api_key=HF_TOKEN)

MODEL_ID = None   # resolved on the first successful call


def chat(system: str, user: str, max_tokens: int = 300, temperature: float = 0.2) -> str:
    """Send one system+user prompt to a free LLM and return plain text.

    Falls back through MODEL_CANDIDATES, then to MOCK_MODE, so a class demo never dies.
    """
    global MODEL_ID
    if MOCK_MODE:
        return _mock(system, user)

    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    tried = ([MODEL_ID] if MODEL_ID else []) + [m for m in MODEL_CANDIDATES if m != MODEL_ID]
    last_error = None
    for model in tried:
        try:
            resp = _client.chat.completions.create(
                model=model, messages=messages,
                max_tokens=max_tokens, temperature=temperature,
            )
            MODEL_ID = model                       # remember what worked
            return (resp.choices[0].message.content or "").strip()
        except Exception as exc:                   # gated / cold / rate-limited
            last_error = exc
            print(f"  [chat] {model} unavailable ({type(exc).__name__}), trying next...")
    print(f"  [chat] all models failed ({last_error}). Falling back to a mock reply.")
    return _mock(system, user)

In [ ]:
# Smoke test - run this before going further.
print(chat("You are a terse assistant.", "Reply with exactly the word: ready"))
print("model in use:", MODEL_ID or "MOCK")

---
## Step 2 — Design the state  *(slide 4)*

State is the shared whiteboard. Ask one question per field:

> *If a second node writes this key, should it **replace** the old value or **add** to it?*

- **Replace** (the default) → facts and decisions: `category`, `urgency`, `escalated`.
- **Add** → logs that several agents contribute to: `findings`, `trace`.
  Mark those with `Annotated[list[str], operator.add]`.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict


class SupportState(TypedDict):
    # ---- inputs -------------------------------------------------------------
    ticket: str

    # ---- facts: last writer wins (default reducer = overwrite) ---------------
    category: str          # "billing" | "technical" | "other"
    urgency: str           # "low" | "medium" | "high"
    escalated: bool
    hops: int              # guard against agents ping-ponging forever

    # ---- logs: every agent APPENDS (operator.add concatenates lists) ---------
    findings: Annotated[list[str], operator.add]
    trace: Annotated[list[str], operator.add]

    # ---- output -------------------------------------------------------------
    final_reply: str


def new_ticket(text: str) -> SupportState:
    """A complete starting state. Fill every key so nodes never hit a KeyError."""
    return {
        "ticket": text, "category": "other", "urgency": "low",
        "escalated": False, "hops": 0,
        "findings": [], "trace": [], "final_reply": "",
    }

print(new_ticket("I was charged twice."))

---
## Step 3 — Write the role prompts  *(slide 9)*

Every prompt follows the same six-part skeleton:

**ROLE** · **SCOPE** (including what is *not* yours) · **INPUT** · **RULES** · **OUTPUT** · **HANDOFF**

Two details do real work here:

- `CONFIDENCE: HIGH|MEDIUM|LOW` as the **last line** — turns prose into a signal we can route on.
- An explicit anti-scope (*"You do NOT handle…"*) — small models drift badly without it.

**Non-coders: this is your cell.** Change the wording, re-run, and watch the behaviour change.

In [ ]:
CLASSIFIER_PROMPT = """ROLE: You are a support ticket classifier.

SCOPE: You only classify. You never answer the customer.

INPUT: One customer ticket.

RULES:
  - category must be exactly one of: billing, technical, other
  - urgency must be exactly one of: low, medium, high
  - "high" means: money is at risk, the customer is blocked, or they ask for a human.

OUTPUT: Reply with ONE line of raw JSON and nothing else.
        {"category": "...", "urgency": "..."}"""


BILLING_PROMPT = """ROLE: You are the Billing specialist on a support team.

SCOPE: Invoices, refunds, duplicate charges, plans, payment failures.
       You do NOT handle logins, bugs, crashes or outages.

INPUT: One customer ticket.

RULES:
  - Refund policy: a duplicate charge is fully refundable within 30 days.
  - Never promise a specific date or amount. State what the policy allows.

OUTPUT: At most 3 sentences, then a final line exactly:
        CONFIDENCE: HIGH or CONFIDENCE: MEDIUM or CONFIDENCE: LOW"""


TECH_PROMPT = """ROLE: You are the Technical support specialist on a support team.

SCOPE: Bugs, crashes, logins, performance, outages.
       You do NOT handle refunds, invoices or pricing.

INPUT: One customer ticket.

RULES:
  - Give concrete steps the customer can take right now.
  - If you would need account access or logs you do not have, say CONFIDENCE: LOW.

OUTPUT: At most 3 sentences, then a final line exactly:
        CONFIDENCE: HIGH or CONFIDENCE: MEDIUM or CONFIDENCE: LOW"""


ESCALATION_PROMPT = """ROLE: You are the Escalation manager.

SCOPE: You prepare a handover for a human agent. You do not talk to the customer.

INPUT: The ticket plus everything the specialists found.

RULES:
  - Be factual. Do not invent account details.

OUTPUT: 3 short lines - Summary, Recommended action, Target response time."""


RESPONDER_PROMPT = """ROLE: You are the Responder. You write the final reply that the CUSTOMER reads.

SCOPE: Rewriting internal findings into one clear message.

INPUT: The ticket plus the internal findings.

RULES:
  - Warm, plain English. No internal notes, no confidence labels, no agent names.
  - Do not invent anything that is not in the findings.

OUTPUT: At most 120 words. No greeting placeholders like [Name]."""

print("prompts defined")

---
## Step 4 — Build the first node: `triage`  *(slide 5)*

Every node follows the same **four beats**:

1. **READ** only the keys it needs
2. **DO** exactly one thing
3. **VALIDATE** — never trust free text from an LLM
4. **RETURN** a *partial* update (only the keys it changed)

Beat 3 is the one beginners skip. A model asked for JSON will happily return
`` `Sure! Here you go: {"category": "Billing"}` `` — with a code fence, a capital B, or a
category you never defined. `extract_json` + an allow-list turns that into something safe to route on.

In [ ]:
import json, re


def extract_json(text: str) -> dict:
    """Pull the first {...} block out of an LLM reply. Never raises."""
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


# Quick check that it survives messy model output
print(extract_json('Sure! Here is the JSON:\n```json\n{"category": "Billing"}\n```'))
print(extract_json('I am not able to answer that.'))

In [ ]:
CATEGORIES = {"billing", "technical", "other"}
URGENCIES  = {"low", "medium", "high"}


def triage_node(state: SupportState) -> dict:
    """Classify the ticket. Does NOT answer it."""

    # 1. READ
    ticket = state["ticket"]

    # 2. DO - one LLM call, nothing else
    raw = chat(CLASSIFIER_PROMPT, ticket, max_tokens=60)

    # 3. VALIDATE - coerce into the allow-list, never crash
    data     = extract_json(raw)
    category = str(data.get("category", "other")).strip().lower()
    urgency  = str(data.get("urgency", "low")).strip().lower()
    if category not in CATEGORIES:
        category = "other"
    if urgency not in URGENCIES:
        urgency = "low"

    # 4. RETURN a partial update
    return {
        "category": category,
        "urgency": urgency,
        "trace": [f"triage        -> category={category}, urgency={urgency}"],
    }

In [ ]:
# A node is just (dict) -> dict, so you can test it with NO graph at all.
demo = new_ticket("I was charged twice for my subscription this month!")
print(triage_node(demo))

---
## Step 5 — Communication protocol #1: shared state  *(slide 10)*

Notice what just happened: `triage_node` wrote `category` and `urgency` into the state.
It did not call any other agent. The next agent will simply **read** those keys.

That is the whole protocol — agents talk by reading and writing one shared object.
It is the default in LangGraph and the one to reach for first, because at any moment you
can print the state and know everything the system knows.

---
## Step 6 — Communication protocol #2: a conditional edge  *(slide 7)*

The **LLM classifies**. **Plain Python routes.** Keeping these separate is what stops a
stray `"Billing."` from crashing your graph.

Note this router is a normal function — free, instant, and unit-testable.

In [ ]:
def route_after_triage(state: SupportState) -> Literal["billing_agent", "tech_agent", "escalation_agent"]:
    """Decide which specialist handles this ticket."""
    # Anything urgent that no specialist owns goes straight to a human.
    if state["category"] == "other":
        return "escalation_agent"
    if state["category"] == "billing":
        return "billing_agent"
    return "tech_agent"


# Test the routing policy on its own - no LLM, no graph, instant.
assert route_after_triage({**new_ticket("x"), "category": "billing"})   == "billing_agent"
assert route_after_triage({**new_ticket("x"), "category": "technical"}) == "tech_agent"
assert route_after_triage({**new_ticket("x"), "category": "other"})     == "escalation_agent"
print("router OK")

---
## Step 7 — Communication protocol #3: handoff with `Command`  *(slide 11)*

The specialists decide their **own** successor, because the decision depends on what they
just discovered (their confidence). `Command` does two things in one atomic return:

```python
Command(goto="escalation_agent",              # the edge
        update={"findings": [...]})           # the state change
```

Because the agent carries its own edge, we will **not** add an `add_edge` out of these nodes.

The `hops` counter is the guard from slide 11: two agents can ping-pong forever, so we count.

In [ ]:
from langgraph.types import Command

MAX_HOPS = 3


def _confidence(text: str) -> str:
    up = text.upper()
    for level in ("LOW", "MEDIUM", "HIGH"):
        if f"CONFIDENCE: {level}" in up:
            return level
    return "LOW"          # no label at all -> treat as unsure


def billing_agent(state: SupportState) -> Command[Literal["escalation_agent", "responder"]]:
    answer = chat(BILLING_PROMPT, state["ticket"])
    conf   = _confidence(answer)

    needs_human = conf == "LOW" or state["urgency"] == "high"
    if state["hops"] >= MAX_HOPS:          # safety valve
        needs_human = True
    goto = "escalation_agent" if needs_human else "responder"

    return Command(
        goto=goto,
        update={
            "hops": state["hops"] + 1,
            "findings": [f"[billing] {answer}"],
            "trace": [f"billing_agent -> {goto} (confidence={conf})"],
        },
    )


def tech_agent(state: SupportState) -> Command[Literal["escalation_agent", "responder"]]:
    answer = chat(TECH_PROMPT, state["ticket"])
    conf   = _confidence(answer)

    needs_human = conf == "LOW" or state["urgency"] == "high"
    if state["hops"] >= MAX_HOPS:
        needs_human = True
    goto = "escalation_agent" if needs_human else "responder"

    return Command(
        goto=goto,
        update={
            "hops": state["hops"] + 1,
            "findings": [f"[tech] {answer}"],
            "trace": [f"tech_agent    -> {goto} (confidence={conf})"],
        },
    )

---
## Step 8 — The last two roles: escalation and responder  *(slide 8)*

- **Escalation / Human** — takes over when confidence is low or stakes are high.
  In a real system this is where `interrupt()` would pause for an actual person.
- **Aggregator / Responder** — merges everything into one message in one voice.
  Having a single node own the customer-facing text is how you stop internal notes leaking out.

In [ ]:
def escalation_agent(state: SupportState) -> dict:
    context = state["ticket"] + "\n\nInternal findings:\n" + "\n".join(state["findings"] or ["(none)"])
    summary = chat(ESCALATION_PROMPT, context, max_tokens=200)
    return {
        "escalated": True,
        "findings": [f"[escalation] {summary}"],
        "trace": ["escalation    -> responder (human handover prepared)"],
    }


def responder(state: SupportState) -> dict:
    context = state["ticket"] + "\n\nInternal findings:\n" + "\n".join(state["findings"] or ["(none)"])
    reply   = chat(RESPONDER_PROMPT, context, max_tokens=250)
    return {
        "final_reply": reply,
        "trace": ["responder     -> END"],
    }

---
## Step 9 — Wire the graph  *(slide 3)*

The same four steps as every LangGraph program:

1. `StateGraph(SupportState)` — declare the state
2. `add_node` — add the work
3. `add_edge` / `add_conditional_edges` — wire the flow
4. `compile()` — get something runnable

Watch which edges we do **not** write: nothing leaves `billing_agent` or `tech_agent`,
because their `Command` return carries the edge.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(SupportState)

# 2. nodes
builder.add_node("triage",           triage_node)
builder.add_node("billing_agent",    billing_agent)
builder.add_node("tech_agent",       tech_agent)
builder.add_node("escalation_agent", escalation_agent)
builder.add_node("responder",        responder)

# 3. edges
builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", route_after_triage)   # the graph routes
# (no edges out of billing_agent / tech_agent - Command routes)
builder.add_edge("escalation_agent", "responder")
builder.add_edge("responder", END)

# 4. compile - the checkpointer gives us memory + resume (Step 11)
graph = builder.compile(checkpointer=InMemorySaver())
print("compiled")

In [ ]:
# See what you actually built. Great sanity check - and great documentation.
print(graph.get_graph().draw_mermaid())

try:                                    # renders an image in Colab / Jupyter
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("(diagram image unavailable, the text above is the same graph)", exc)

---
## Step 10 — Run it

`run_ticket` invokes the graph and prints the **trace** — the line every node appended.
The trace is your whole debugging toolkit: it shows the exact path the ticket took.

In [ ]:
import itertools
_thread_ids = itertools.count(1)


def run_ticket(text: str, thread_id: str | None = None) -> SupportState:
    thread_id = thread_id or f"ticket-{next(_thread_ids)}"
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 12}

    print("=" * 78)
    print("TICKET :", text)
    print("THREAD :", thread_id)

    result = graph.invoke(new_ticket(text), config=config)

    print("-" * 78)
    print("PATH TAKEN")
    for line in result["trace"]:
        print("   ", line)
    print("-" * 78)
    print(f"category={result['category']}  urgency={result['urgency']}  "
          f"escalated={result['escalated']}  hops={result['hops']}")
    print("-" * 78)
    print("REPLY TO CUSTOMER:\n")
    print(result["final_reply"])
    print("=" * 78, "\n")
    return result

In [ ]:
# Path 1 - billing, high urgency -> specialist hands off to escalation
_ = run_ticket("I was charged twice for my Pro subscription this month. I want a refund!")

In [ ]:
# Path 2 - technical, model is usually confident -> straight to the responder
_ = run_ticket("The app crashes every time I open the settings screen on Android 14.")

In [ ]:
# Path 3 - no specialist owns this -> the router sends it straight to a human
_ = run_ticket("I would like to speak to a manager about my experience last week.")

### Read the traces above

Three different tickets, three different paths, one graph. Two different mechanisms decided
those paths — and knowing *which one fired where* is the whole lesson:

| Decision | Made by | Why there |
|---|---|---|
| Which specialist gets the ticket | `route_after_triage` (conditional edge) | It is company **policy** — visible, testable, outside the agent |
| Whether to escalate | `Command` inside the agent | It depends on what the agent **just discovered** |

---
## Step 11 — Memory: threads and the checkpointer  *(slide 14)*

We compiled with `checkpointer=InMemorySaver()`. State is now saved after every step,
keyed by `thread_id`:

- **same** `thread_id` → the graph remembers that conversation
- **new** `thread_id` → a fresh run

`get_state` reads the latest checkpoint; `get_state_history` lets you rewind ("time travel"),
which is a genuinely useful debugging tool.

> `InMemorySaver` disappears when the kernel stops. For production, use a persistent
> saver such as `langgraph-checkpoint-postgres`.

In [ ]:
cfg = {"configurable": {"thread_id": "demo-thread"}}
_ = run_ticket("My invoice shows a charge I do not recognise.", thread_id="demo-thread")

snapshot = graph.get_state(cfg)
print("saved keys      :", sorted(snapshot.values.keys()))
print("next node to run:", snapshot.next, "  (empty tuple = finished)")
print("checkpoints kept:", len(list(graph.get_state_history(cfg))))

---
## Step 12 — Your exercises

Work top to bottom. Each one changes exactly one idea from the session.

**1 — Add a `shipping` specialist** *(agent roles)*
   - add `"shipping"` to `CATEGORIES` and to the classifier prompt
   - write a `SHIPPING_PROMPT` using the same six-part skeleton
   - write `shipping_agent` returning a `Command` (copy `billing_agent`)
   - add the node, and add a branch in `route_after_triage`
   - rebuild the graph and test with *"My order has not arrived after 3 weeks."*

**2 — Escalate on medium urgency too** *(routing policy)*
   - in `billing_agent`, change `state["urgency"] == "high"` to `in ("high", "medium")`
   - rerun the billing ticket and compare the traces

**3 — Add a critic before the responder** *(a new role + a loop)*
   - write `critic_node` that grades `findings` and returns `PASS` or `FAIL: <reason>`
   - on `FAIL`, `Command(goto=...)` back to the specialist — **and cap it** with `hops`
   - this is the classic writer/critic loop; watch how fast it can run away without the cap

**4 — Break it on purpose** *(why validation matters)*
   - delete the allow-list check in `triage_node` and make `CLASSIFIER_PROMPT` return
     `"Billing."` instead of JSON
   - run it, read the error, then put the validation back

Scaffolding for exercise 1 is in the next cell.

In [ ]:
# ── EXERCISE 1 SCAFFOLD ───────────────────────────────────────────────────────
# Fill in the TODOs, then run this cell and the one after it.

SHIPPING_PROMPT = """ROLE: You are the Shipping specialist on a support team.

SCOPE: Deliveries, tracking, delays, damaged parcels, address changes.
       You do NOT handle refunds, invoices, bugs or logins.

INPUT: One customer ticket.

RULES:
  - TODO: write two grounding rules (e.g. standard delivery is 5-7 working days).

OUTPUT: At most 3 sentences, then a final line exactly:
        CONFIDENCE: HIGH or CONFIDENCE: MEDIUM or CONFIDENCE: LOW"""


def shipping_agent(state: SupportState) -> Command[Literal["escalation_agent", "responder"]]:
    answer = chat(SHIPPING_PROMPT, state["ticket"])
    conf   = _confidence(answer)
    goto   = "escalation_agent" if (conf == "LOW" or state["urgency"] == "high") else "responder"
    return Command(
        goto=goto,
        update={"hops": state["hops"] + 1,
                "findings": [f"[shipping] {answer}"],
                "trace": [f"shipping_agent-> {goto} (confidence={conf})"]},
    )


def route_v2(state: SupportState) -> Literal["billing_agent", "tech_agent",
                                             "shipping_agent", "escalation_agent"]:
    # TODO: add the "shipping" branch here
    if state["category"] == "billing":
        return "billing_agent"
    if state["category"] == "technical":
        return "tech_agent"
    return "escalation_agent"


print("scaffold loaded - now add 'shipping' to CATEGORIES and to CLASSIFIER_PROMPT")

In [ ]:
# ── EXERCISE 1: rebuild the graph with the new specialist ─────────────────────
b2 = StateGraph(SupportState)
b2.add_node("triage",           triage_node)
b2.add_node("billing_agent",    billing_agent)
b2.add_node("tech_agent",       tech_agent)
b2.add_node("shipping_agent",   shipping_agent)
b2.add_node("escalation_agent", escalation_agent)
b2.add_node("responder",        responder)

b2.add_edge(START, "triage")
b2.add_conditional_edges("triage", route_v2)
b2.add_edge("escalation_agent", "responder")
b2.add_edge("responder", END)

graph_v2 = b2.compile(checkpointer=InMemorySaver())
print(graph_v2.get_graph().draw_mermaid())

# out = graph_v2.invoke(new_ticket("My order has not arrived after 3 weeks."),
#                       config={"configurable": {"thread_id": "ex1"}})
# print("\n".join(out["trace"]))

---
## Recap — the whole session in one page

| Idea | The rule |
|---|---|
| **Node** | READ → DO ONE THING → VALIDATE → RETURN a partial dict |
| **State** | Overwrite for facts, `Annotated[list, operator.add]` for logs. Keep it small |
| **Role** | ROLE / SCOPE / INPUT / RULES / OUTPUT / HANDOFF. One job per agent |
| **Route** | The LLM classifies, Python routes. Always check against an allow-list |
| **Handoff** | `Command(goto=..., update=...)` — carry context, and cap the hops |
| **Control** | A counter *and* a maximum on every loop. A trace line in every node |

### Where to go next

- **Subgraphs** — compile a graph and use it as a node when one step is itself multi-step
- **`interrupt()`** — pause for a real human before sending the reply
  (put any email/payment in its own node *after* the approval, or it fires twice on resume)
- **Persistent checkpointers** — `langgraph-checkpoint-postgres` instead of `InMemorySaver`
- **`langgraph-supervisor`** — the prebuilt supervisor, once you understand what it generates
- **Streaming** — `graph.stream(...)` to push state updates into a UI as they happen

Docs: <https://docs.langchain.com/oss/python/langgraph/overview>